# Trizzy Writer ? WAN 2.1 Colab Worker

Reusable Google Colab GPU-worker foundation for video rendering from Trizzy Writer.

This initial notebook checks the GPU, mounts Google Drive, creates the worker folders, and defines the render-job format. WAN installation and inference cells will be expanded as the integration is completed.

## 1. Runtime and GPU check

In [ ]:
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime in Colab: Runtime > Change runtime type > GPU.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Create reusable worker folders

In [ ]:
from pathlib import Path

WORKER_ROOT = Path('/content/drive/MyDrive/TrizzyWriter/video-worker')
QUEUE_DIR = WORKER_ROOT / 'queue'
PROCESSING_DIR = WORKER_ROOT / 'processing'
OUTPUT_DIR = WORKER_ROOT / 'outputs'
FAILED_DIR = WORKER_ROOT / 'failed'
MODEL_CACHE_DIR = WORKER_ROOT / 'models'

for folder in [QUEUE_DIR, PROCESSING_DIR, OUTPUT_DIR, FAILED_DIR, MODEL_CACHE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Worker root:", WORKER_ROOT)
print("Queue:", QUEUE_DIR)
print("Outputs:", OUTPUT_DIR)

## 4. Render-job schema

Trizzy Writer will eventually create JSON files in the queue folder using this structure.

In [ ]:
import json
import uuid
from datetime import datetime, timezone

example_job = {
    'id': str(uuid.uuid4()),
    'created_at': datetime.now(timezone.utc).isoformat(),
    'task': 'text-to-video',
    'model': 'Wan-AI/Wan2.1-T2V-1.3B',
    'prompt': 'A cinematic nighttime performance in a rain-soaked Memphis street, realistic lighting, slow camera push-in.',
    'negative_prompt': 'blurry, distorted face, extra fingers, duplicate people, text, watermark',
    'aspect_ratio': '9:16',
    'duration_seconds': 5,
    'fps': 16,
    'seed': 12345,
    'reference_image': None,
}

print(json.dumps(example_job, indent=2))

## 5. Write a test job

Run this cell to place a sample request in Google Drive. It does not render yet.

In [ ]:
job_path = QUEUE_DIR / f"{example_job['id']}.json"
job_path.write_text(json.dumps(example_job, indent=2), encoding='utf-8')
print("Queued test job:", job_path)

## Next implementation stage

- Install the official WAN 2.1 runtime and pinned dependencies.
- Cache the model in Google Drive.
- Process queued JSON jobs.
- Generate MP4 files and status JSON.
- Add image-to-video support.
- Connect the queue to Trizzy Writer's Video Director interface.